In [ ]:
# mount the data from google drive
from google.colab import drive
from google.colab import files
drive.mount("/content/drive")

# navigate to the project folder
%cd drive/MyDrive/projects/CVPR

Mounted at /content/drive
/content/drive/MyDrive/projects/CVPR


In [ ]:
#### not parallel code

import numpy as np
from skimage.transform import resize
import pandas as pd
import cv2
from scipy.ndimage import label, zoom, center_of_mass
from tqdm.auto import tqdm
import multiprocessing as mp
import os

def get_common_files(fixation_dir, depth_dir):
    """
    Get common file names between two directories
    """
    fixation_files = {os.path.splitext(file)[0] for file in os.listdir(fixation_dir)}
    depth_files = {os.path.splitext(file)[0] for file in os.listdir(depth_dir)}

    common_files = list(fixation_files.intersection(depth_files))

    print(f"Total common files: {len(common_files)}")
    print("Sample common files:", common_files[:10])

    return common_files
def list_files(path):
    """Lists all files in the specified directory."""

    files = []
    for entry in os.listdir(path):
        if os.path.isfile(os.path.join(path, entry)):
            files.append(entry[:-4])
    return files

def calculate_distance(mask):
    centroid = np.array(center_of_mass(mask))
    centroid = (int(centroid[0]), int(centroid[1]))
    sum_sq = np.sum(np.square(centroid - np.array((525,840))))
    distance = np.sqrt(sum_sq)
    return centroid, distance

def calculate_depth_at_center(mask, depth_map):
    centroid = center_of_mass(mask)
    depth = depth_map[int(centroid[0]), int(centroid[1])]
    return depth

def get_prominent_values(mask_array, threshold_percent=.5):
    """
    Find values that are more than a specified percentage in the array

    Parameters:
    -----------
    mask_array : numpy.ndarray
        Input array to analyze
    threshold_percent : float, optional
        Minimum percentage threshold (default is 1)

    Returns:
    --------
    tuple: (values above threshold, counts, percentages)
    """
    # Get unique values and their counts
    unique_values, counts = np.unique(mask_array, return_counts=True)

    # Calculate percentages
    total_elements = mask_array.size
    percentages = (counts / total_elements) * 100

    # Filter values above the threshold
    threshold_mask = percentages > threshold_percent

    prominent_values = unique_values[threshold_mask]
    prominent_counts = counts[threshold_mask]
    prominent_percentages = percentages[threshold_mask]

    # Sort in descending order of percentages
    sort_indices = np.argsort(prominent_percentages)[::-1]

    return prominent_values[sort_indices]

def compare_directory_files(dir_a, dir_b):
    """
    Compare file names in two directories, ignoring file extensions.

    Args:
    dir_a (str): Path to the first directory
    dir_b (str): Path to the second directory

    Returns:
    list: Files found in dir_a that are not in dir_b
    """
    # Get file names from directory A (without extensions)
    files_a = {os.path.splitext(file)[0] for file in os.listdir(dir_a)}

    # Get file names from directory B (without extensions)
    files_b = {os.path.splitext(file)[0] for file in os.listdir(dir_b)}

    # Find files in A that are not in B
    return files_a, files_b, list(files_a - files_b)

def resize_labeled_array(labels, new_height=1050, new_width=1680, interpolation='nearest'):
    """
    Resize a labeled array while preserving label integrity.

    Parameters:
    -----------
    labels : numpy.ndarray
        The input labeled array
    new_height : int, optional
        Desired height of the output array (default: 1050)
    new_width : int, optional
        Desired width of the output array (default: 1680)
    interpolation : str, optional
        Interpolation method. 'nearest' is crucial for label preservation

    Returns:
    --------
    numpy.ndarray
        Resized labeled array
    """
    # Resize using skimage with nearest neighbor interpolation
    resized_labels = resize(
        labels,
        (new_height, new_width),
        order=0,  # Nearest neighbor interpolation
        preserve_range=True,  # Preserve original label values
        anti_aliasing=False   # Avoid blurring labels
    ).astype(labels.dtype)

    return resized_labels

def filter_labels_by_size(labels, min_size):
    """
    Filter out labels (clusters) that are smaller than min_size pixels
    by incorporating them into adjacent clusters.

    Parameters:
    -----------
    labels : numpy.ndarray
        The labeled array from scipy.ndimage.label()
    min_size : int
        Minimum number of pixels required for a cluster to be kept

    Returns:
    --------
    numpy.ndarray
        Filtered labels array with small clusters incorporated into neighbors
    """
    # Count the size of each label
    label_sizes = np.bincount(labels.ravel())

    # Create a mask of labels that are too small
    small_labels = np.where((label_sizes < min_size) & (np.arange(len(label_sizes)) > 0))[0]

    # Create a copy of the labels array
    filtered_labels = labels.copy()

    # For each small label, incorporate it into a neighbor
    for label in small_labels:
        # Get positions of current small cluster
        small_cluster_pos = np.where(labels == label)

        # Initialize list to track neighboring labels and their frequencies
        neighbor_labels = []

        # For each pixel in the small cluster
        for i in range(len(small_cluster_pos[0])):
            r, c = small_cluster_pos[0][i], small_cluster_pos[1][i]

            # Check 8 neighbors (if within bounds)
            for dr in [-1, 0, 1]:
                for dc in [-1, 0, 1]:
                    if dr == 0 and dc == 0:
                        continue  # Skip the pixel itself

                    # Calculate neighbor coordinates
                    nr, nc = r + dr, c + dc

                    # Check if within bounds
                    if (0 <= nr < labels.shape[0] and 0 <= nc < labels.shape[1]):
                        neighbor_label = labels[nr, nc]
                        # Only consider non-background and non-small neighbors
                        if neighbor_label > 0 and neighbor_label != label and neighbor_label not in small_labels:
                            neighbor_labels.append(neighbor_label)

        # If we found valid neighbors, assign to the most common neighboring label
        if neighbor_labels:
            from collections import Counter
            most_common_neighbor = Counter(neighbor_labels).most_common(1)[0][0]
            filtered_labels[labels == label] = most_common_neighbor
        else:
            # If no valid neighbors, keep the small cluster as is
            # or you could set to background as a fallback if preferred
            pass

    return filtered_labels

def process_chunk(chunk):
  out_file = str(chunk[0]) + "coco-values-parallel.csv"
  values_file_path = os.path.join(output_dir, out_file)
  ids_master = [1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  110,
  111,
  112,
  118,
  120,
  121,
  122,
  124,
  126,
  127,
  128,
  129,
  130,
  131,
  132,
  133,
  134,
  135,
  136,
  137,
  138,
  140,
  141,
  142,
  143,
  144,
  145,
  146,
  147,
  149,
  150,
  151,
  152,
  153,
  154,
  155,
  157,
  158,
  159,
  160,
  161,
  162,
  163,
  164,
  165,
  166,
  167,
  168,
  169,
  178,
  179,
  180,
  181]
  with open(values_file_path, 'w') as values_file:
        values_file.write("image,label,sum,size,ecc,id,log_size,log_sum,centroid_x,centroid_y,dg_center,depth_center,num_clusters, num_cluster, unique_id\n")
        # Loop through images
        for image in chunk:
            # Load fixation array
            fixation_array = np.load(fixation_dir + image + '.npy')

            # Load depth map
            depth_map = np.load(depth_dir + image + '.npy')

            # Load deepgaze map & resize
            dg_map = np.load(dg_dir + image + '.npy')
            if dg_map is None or dg_map.size == 0:
                raise ValueError(f"Error: DeepGaze map is empty or invalid for {image}")
            dg_map = np.nan_to_num(dg_map, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
            dg_map = cv2.resize(dg_map, (1680, 1050), interpolation=cv2.INTER_CUBIC)

            # Load image mask
            img_array = np.load(mask_dir + image + '.npy')
            if img_array.ndim == 3 and img_array.shape[-1] == 3:
                img_array = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
            elif img_array.ndim > 2:
                raise ValueError(f"Array in {image} has too many dimensions: {img_array.ndim}")

            # Normalize if needed
            if img_array.dtype in [np.float32, np.float64] and img_array.max() <= 1.0:
                img_array = (img_array * 255).astype(np.uint8)
               # Get unique mask IDs (excluding background)

            values = get_prominent_values(img_array, threshold_percent=.05)
            values = values[values>0]
            mask_ids = [x for x in values if x in ids_master]

            if len(mask_ids) > 0:
              for mask_id in mask_ids:
                mask = img_array == mask_id

                # Resize mask
                zoom_factor = (35 / mask.shape[0], 56 / mask.shape[1])
                resized_arr = zoom(mask, zoom_factor, order=0)

                # Find clusters
                labeled_array, num_clusters = label(resized_arr, structure=structure)


                labeled_array = filter_labels_by_size(labeled_array, min_size=4)
                labeled_array = resize_labeled_array(labeled_array)

                cluster_labels = np.unique(labeled_array)[np.unique(labeled_array) != 0]
                #cluster_labels = get_prominent_values(labeled_array, threshold_percent=.15)
                cluster_labels = cluster_labels[cluster_labels>0]
                for num_cluster, cluster_label in enumerate(cluster_labels):
                  num_clusters = len(cluster_labels)
                  mask = labeled_array == cluster_label

                          # Compute centroid & eccentricity
                  centroid, distance = calculate_distance(mask)

                              # Compute values
                  fixation_sum = np.sum(fixation_array[mask])
                  size = np.count_nonzero(mask)
                  depth_center = depth_map[centroid]
                  dg_center = dg_map[centroid]
                  label_name = list(guide[guide['id'] == mask_id + 1]['label'])[0]
                  unique_id = image + label_name + str(num_cluster)

                          # Write to file
                  values_file.write(
                      f"{image},{label_name},{fixation_sum},{size},{distance},{mask_id},"
                      f"{np.log(size)},{np.log(fixation_sum)},{centroid[1]},{centroid[0]},"
                      f"{dg_center},{depth_center}, {num_clusters}, {num_cluster}, {unique_id}\n"
                  )


  result = 'hey'

  return result

output_dir = "/content/drive/MyDrive/projects/CVPR"  # Change to your preferred folder
os.makedirs(output_dir, exist_ok=True)  # Ensure the folder exists
fixation_dir = 'ms-coco/fixations/'
img_dir = 'ms-coco/coco-images/'
mask_dir = 'ms-coco/modified_numpy_arrays/'
depth_dir = 'ms-coco/depth/'
dg_dir = 'ms-coco/deepgaze/'
image_list = get_common_files(fixation_dir, depth_dir)
structure = np.array([[0,1,0], [1,1,1], [0,1,0]])  # 4-way connectivity
chunks = list(np.array_split(image_list, 12))

guide = pd.read_csv('ms-coco/labels.txt', sep=':', header=None, names=['id', 'label'], skipinitialspace=True)
background_ids = [101, 102, 103, 114, 115, 116, 117, 118, 120, 124, 126, 140, 149, 157, 171, 172, 173, 174, 175, 176, 177, 178, 183]
ids_master = [value for value in guide['id'] if value not in background_ids]
ids_master = [x - 1 for x in ids_master if x > 1]  # Adjust IDs to match your mask indexing

# Create a pool of workers
num_processes = mp.cpu_count()  # Use all available CPU cores
pool = mp.Pool(processes=num_processes)

# Use tqdm with imap to show progress
results = list(tqdm(pool.imap(process_chunk, chunks), total=len(chunks), desc="Processing chunks"))

# Clean up resources
pool.close()
pool.join()






Total common files: 2243
Sample common files: ['000000530097', '000000460460', '000000476642', '000000511425', '000000495867', '000000490311', '000000516668', '000000309368', '000000342452', '000000365351']


Processing chunks:   0%|          | 0/12 [00:00<?, ?it/s]

<ipython-input-2-c6eaa6261526>:428: RuntimeWarning: divide by zero encountered in log
  f"{np.log(size)},{np.log(fixation_sum)},{centroid[1]},{centroid[0]},"
<ipython-input-2-c6eaa6261526>:428: RuntimeWarning: divide by zero encountered in log
  f"{np.log(size)},{np.log(fixation_sum)},{centroid[1]},{centroid[0]},"
<ipython-input-2-c6eaa6261526>:428: RuntimeWarning: divide by zero encountered in log
  f"{np.log(size)},{np.log(fixation_sum)},{centroid[1]},{centroid[0]},"
<ipython-input-2-c6eaa6261526>:428: RuntimeWarning: divide by zero encountered in log
  f"{np.log(size)},{np.log(fixation_sum)},{centroid[1]},{centroid[0]},"
<ipython-input-2-c6eaa6261526>:428: RuntimeWarning: divide by zero encountered in log
  f"{np.log(size)},{np.log(fixation_sum)},{centroid[1]},{centroid[0]},"
<ipython-input-2-c6eaa6261526>:428: RuntimeWarning: divide by zero encountered in log
  f"{np.log(size)},{np.log(fixation_sum)},{centroid[1]},{centroid[0]},"
<ipython-input-2-c6eaa6261526>:428: RuntimeWarning: 

In [ ]:
import glob
import os
import pandas as pd

def combine_csv_files(folder_path, output_file='combined_output3.csv'):
    """
    Combines all CSV files in the specified folder into one CSV file.

    Parameters:
    folder_path (str): Path to the folder containing CSV files
    output_file (str): Name of the output file

    Returns:
    bool: True if successful, False otherwise
    """
    try:
        # Get all CSV files in the folder
        csv_files = glob.glob(os.path.join(folder_path, '*.csv'))

        if not csv_files:
            print(f"No CSV files found in {folder_path}")
            return False

        # Create an empty list to store individual dataframes
        dfs = []

        # Read each CSV file and append to the list
        for file in csv_files:
            df = pd.read_csv(file)
            dfs.append(df)
            print(f"Processed: {file}")

        # Combine all dataframes into one
        combined_df = pd.concat(dfs, ignore_index=True)

        # Write the combined dataframe to a new CSV file
        combined_df.to_csv(output_file, index=False)

        print(f"Successfully combined {len(csv_files)} CSV files into {output_file}")
        return True

    except Exception as e:
        print(f"Error: {e}")
        return False

# Example usage
if __name__ == "__main__":
    # Replace with your folder path
    folder_path = "./ms-coco/coco-results3"
    combine_csv_files(folder_path)

Processed: ./ms-coco/coco-results3/000000052151coco-values-parallel.csv
Processed: ./ms-coco/coco-results3/000000433607coco-values-parallel.csv
Processed: ./ms-coco/coco-results3/000000514280coco-values-parallel.csv
Processed: ./ms-coco/coco-results3/000000237054coco-values-parallel.csv
Processed: ./ms-coco/coco-results3/000000142557coco-values-parallel.csv
Processed: ./ms-coco/coco-results3/000000327768coco-values-parallel.csv
Processed: ./ms-coco/coco-results3/000000531552coco-values-parallel.csv
Processed: ./ms-coco/coco-results3/000000553283coco-values-parallel.csv
Processed: ./ms-coco/coco-results3/000000121682coco-values-parallel.csv
Processed: ./ms-coco/coco-results3/000000422172coco-values-parallel.csv
Processed: ./ms-coco/coco-results3/000000369778coco-values-parallel.csv
Processed: ./ms-coco/coco-results3/000000563301coco-values-parallel.csv


<ipython-input-2-215a2cf526fb>:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat(dfs, ignore_index=True)


Successfully combined 12 CSV files into combined_output3.csv
